In [2]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
from tqdm import tqdm
import os
import warnings

# Suppress potential warnings from correlation calculations
warnings.filterwarnings('ignore')

ratio_dir = r"C:\Users\28616\Desktop\Sample_Ratio_Matrices"
gene_dir = r"C:\Users\28616\Desktop\gene_processed"
output_dir = r"C:\Users\28616\Desktop\Correlation_Results"
os.makedirs(output_dir, exist_ok=True)

sample_ids = ["49", "50", "51", "52", "53", "54"]

# Outer loop for samples
for s_id in tqdm(sample_ids, desc="Overall Progress"):
    ratio_file = os.path.join(ratio_dir, f"Sample_{s_id}_ratio_matrix.csv")
    gene_file = os.path.join(gene_dir, f"Sample_{s_id}_gene_processed.csv")
    
    if not (os.path.exists(ratio_file) and os.path.exists(gene_file)):
        continue
    
    df_ratio = pd.read_csv(ratio_file, index_col=0)
    df_gene = pd.read_csv(gene_file, index_col=0)
    
    results = []
    
    # Inner loop for sites with progress bar
    # position=1 and leave=False keeps the console clean
    for site in tqdm(df_ratio.columns, desc=f"Processing Sample {s_id}", position=1, leave=False):
        vec_ratio = df_ratio[site]
        
        for gene in df_gene.columns:
            vec_gene = df_gene[gene]
            
            # Check variance to avoid division by zero errors
            if vec_ratio.std() == 0 or vec_gene.std() == 0:
                continue
                
            try:
                s_rho, s_p = spearmanr(vec_gene, vec_ratio)
                p_r, p_p = pearsonr(vec_gene, vec_ratio)
                
                results.append({
                    'Site': site,
                    'Gene': gene,
                    'Spearman_Rho': s_rho,
                    'Spearman_P': s_p,
                    'Pearson_R': p_r,
                    'Pearson_P': p_p
                })
            except:
                continue
                
    if results:
        output_path = os.path.join(output_dir, f"Corr_Results_Sample_{s_id}.csv")
        pd.DataFrame(results).to_csv(output_path, index=False)

print("Correlation analysis complete. Results saved to 'Correlation_Results'.")

Overall Progress: 100%|██████████████████████████████████████████████████████████████████| 6/6 [00:09<00:00,  1.66s/it]

Correlation analysis complete. Results saved to 'Correlation_Results'.
